In [1]:
# pip install Flask

In [1]:
from flask import Flask, jsonify
import pickle
from werkzeug.wrappers import Request, Response
from werkzeug.serving import run_simple

from lime import lime_text
from sklearn.pipeline import make_pipeline
from lime.lime_text import LimeTextExplainer

import re

In [2]:
test_data = ['MessageDigest instance = MessageDigest.getInstance("sha-1");',
             'Log.e("Java/LocalDNSServer", "Failed to create server socket");',
             'builder.addAddress("10.1.10.1", 32);',
             'if (intValue != 0 && !TextUtils.isEmpty("127.0.0.1")) {',
             'import java.util.Random;',
             'public static final String OWM_API_KEY = "82eff2c845841c89c837d4e125613d83";',
             'System.out.println(zone);',
             'import java.security.MessageDigest;',
             'String name = "MyApp";',
             'System.out.println("echo <text>\n\nprints the text\n");']

In [3]:
def preprocess_comments_and_strings(code_line):
    
    processed_code_line = code_line

    encryption_hashing_pattern = "AES|aes|SHA-1|sha-1|SHA1|sha1|MD5|md5"
    ip_pattern = "\w*([0-9]{1,3})\.([0-9]{1,3})\.([0-9]{1,3})\.([0-9]{1,3})\w*"
    string_pattern = "\"[\w|\s|$|&|+|,|:|;|=|?|@|#|_|/|\-|\.|!|`|~|%|\^|\*|\(|\)|\'\\[|\]\{|\}]*\""

    # Checking for encryption related strings
    find_encryption = re.search(encryption_hashing_pattern, processed_code_line)

    # Checking for IP related strings
    find_ip = re.search(ip_pattern, processed_code_line)

    if (find_encryption is None) & (find_ip is None):
        # replacing all strings with dummy string
        processed_code_line = re.sub(string_pattern, "\"user_str\"", processed_code_line)
        
    # replacing comments with dummy comment
    #comment_pattern = "//.*|/\\*(?s:.*?)\\*/|(\"(?:(?<!\\\\)(?:\\\\\\\\)*\\\\\"|[^\r\n\"])*\")"
    comment_pattern = "//.*|/\\*(?s:.*?)\\*/|/\\*(.)*|(.)*\\*/"
    processed_code_line = re.sub(comment_pattern, "//user_comment",processed_code_line)

    return processed_code_line

In [4]:
with open('binary_model.pickle', 'rb') as bin_model_file:
    binary_vectorizer, binary_classifier, binary_xgb_model = pickle.load(bin_model_file)
    
with open('multiclass_model.pickle', 'rb') as multi_model_file:
    multi_vectorizer, mutli_classifier, multi_xgb_model = pickle.load(multi_model_file)

In [5]:
def check_vulnerability(test_code):
    print (test_code)
    xt_binary = binary_vectorizer.transform([test_code])
    is_vulnearble = binary_xgb_model.predict(xt_binary)
    pre_processed_test_code = preprocess_comments_and_strings(test_code)
    vulnerability_probability, vulnerable_code_word_list = show_reason_binary(pre_processed_test_code,is_vulnearble[0])
    vulnerable_class = 0
    is_vulnerable = "Vulnerable Code"
    if(is_vulnearble == 1):
        xt_multi = multi_vectorizer.transform([test_code])        
        vulnerable_class = multi_xgb_model.predict(xt_multi)[0]
#         show_reason_multiclass(pre_processed_test_code,vulnerable_class)
        
    else:
        vulnerable_class = 0
        is_vulnerable = "Non-Vulnerable Code"
    return is_vulnerable, vulnerable_class, vulnerability_probability, vulnerable_code_word_list

In [6]:
def show_reason_binary(test_code, is_vulnerable):
    c = make_pipeline(binary_vectorizer, binary_classifier)
    class_names = [0,1]
    explainer = LimeTextExplainer(class_names=class_names)
#     idx = 1

    exp = explainer.explain_instance(test_code, c.predict_proba, num_features=5, top_labels=1)
#     print(exp.available_labels())
#     print('Document id: %d' % idx)
    vulnerability_probability = c.predict_proba([test_code])[0,1]
    print('Probability(vulnerability) =', vulnerability_probability)
    print('Vulnerable class: ' , is_vulnerable)

#     exp.show_in_notebook(text=True)

    vulnerable_code_word_list = ""
    if(is_vulnerable == 1):
        vulnerable_code_word_list = str(exp.as_list())        

    return vulnerability_probability, vulnerable_code_word_list

In [7]:
app = Flask(__name__)

@app.route('/')
def index():
    return "welcome to vulnerability checker"


@app.route('/code', methods=['GET'])
def get():
    return_string = ""
    for test_code in test_data:
        return_string = return_string + str(check_vulnerability(test_code))
    return return_string


@app.route('/code/<string:test_code>', methods=['GET'])
def getCWE(test_code):
    is_vulnerable, vulnerable_class, vulnerability_probability, vulnerable_code_word_list = check_vulnerability(test_code)
    
    from collections import OrderedDict
    
#     return_string = OrderedDict([('Code',str(test_code)),('Code vulnerability probability',str(vulnerability_probability)),('Code vulnerability Status',str(is_vulnerable)), ('CWE-ID',str(vulnerable_class))])
#     return jsonify(return_string)

    return jsonify({'Code':str(test_code),
                    'Code vulnerability probability':str(vulnerability_probability),
                    'Probability breakdown of vulnerable code words':str(vulnerable_code_word_list),
                    'Code vulnerability Status':str(is_vulnerable),
                    'CWE-ID':str(vulnerable_class)})

    

In [8]:
run_simple('localhost', 5000, app)

 * Running on http://localhost:5000/ (Press CTRL+C to quit)


sdsds


127.0.0.1 - - [28/Jul/2022 19:22:16] "GET /code/sdsds HTTP/1.1" 200 -


Probability(vulnerability) = 0.08488273
Vulnerable class:  0
public static final String OWM_API_KEY = "82eff2c845841c89c837d4e125613d83";


127.0.0.1 - - [28/Jul/2022 19:22:31] "GET /code/public%20static%20final%20String%20OWM_API_KEY%20=%20%2282eff2c845841c89c837d4e125613d83%22; HTTP/1.1" 200 -


...

Probability(vulnerability) = 0.99425936
Vulnerable class:  1
Log.e("JavaLocalDNSServer", "Failed to create server socket");


127.0.0.1 - - [28/Jul/2022 19:26:19] "GET /code/Log.e(%22JavaLocalDNSServer%22,%20%22Failed%20to%20create%20server%20socket%22); HTTP/1.1" 200 -


Probability(vulnerability) = 0.99425936
Vulnerable class:  1
Log.e("JavaLocalDNSServer", "Failed to create server socket");


127.0.0.1 - - [28/Jul/2022 19:26:22] "GET /code/Log.e(%22JavaLocalDNSServer%22,%20%22Failed%20to%20create%20server%20socket%22); HTTP/1.1" 200 -


Probability(vulnerability) = 0.99425936
Vulnerable class:  1
